<a href="https://colab.research.google.com/github/arulbenjaminchandru/ai-engineer-june20/blob/main/Day_9_Embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 9 — What Are Embeddings?
## How AI Finds *Meaning* in Text — Not Just Keywords


### 🎬 The scene we will fix today

It is Monday morning at **Kaveri Insurance**, Chennai.

**Priya** runs the customer support desk. Her help centre has 4,000 articles and a search box. This morning a customer typed:

> **"my hospital bill was not paid"**

The search box returned **zero results**.

But Kaveri Insurance *has* an article about exactly this problem. It is titled:

> **"Cashless request declined — how to claim reimbursement"**

The article is perfect. The search engine could not see it.

Why? Because the customer and the article **used completely different words for the same idea**.

| Customer said | Article says |
|---|---|
| hospital bill | cashless request |
| was not paid | declined |
| — | reimbursement |

**Zero words in common. 100% the same meaning.**

Today you learn the technology that fixes this — **embeddings**. Tomorrow (Day 10) you build it.

---


### 🎯 What you will be able to do by the end

Not "know about". **Do.**

| You will be able to… | Proof you can do it |
|---|---|
| **Explain** what an embedding is to a non-technical manager in 30 seconds | You can say it out loud without notes |
| **Explain** why keyword search fails and semantic search works | You can give the Kaveri example from memory |
| **Compute** cosine similarity by hand and in code | You can do it on a whiteboard |
| **Choose** between MiniLM and MPNet for a real project | You can justify the choice with numbers |
| **Explain** what a vector database adds over a plain list of numbers | You can draw the architecture |
| **Answer** the interview questions in this notebook | You answer them in the mock section |

---

### 🧭 How every section is built (same rhythm every time)

| Block | What it is |
|---|---|
| 🧠 **The idea** | The concept in the simplest possible words |
| 🔬 **How it actually works** | The mechanism, step by step |
| 💻 **Run it** | A short code cell you run yourself |
| 🏢 **In the real world** | A real company, a real number, a real date |
| ⚠️ **Don't mix these up** | The mistakes that get people caught in interviews |
| 🎤 **Interview angle** | A question + a model answer you can say out loud |
| ✅ **Quick check** | A question. Click to reveal the answer. |
| 🤯 **Fun fact** | A verified, real detail |

---

### ⚙️ Setup (run these two cells first)

You need one Colab Secret named **`MY_API_KEY`** holding your Anthropic API key (🔑 icon in the Colab left sidebar).

In [1]:
!pip install -q anthropic sentence-transformers scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 5.2 MB/s eta 0:00:00


In [2]:
# Key + Claude client. Colab Secret must be named MY_API_KEY.
from google.colab import userdata
import os

os.environ["ANTHROPIC_API_KEY"] = userdata.get("MY_API_KEY")

import anthropic
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"

print("Claude is ready ✅")

Claude is ready ✅


---

# Foundations — Four words you need before we start

Read this once. We will use these four words all day.

| Word | Everyday meaning | Kaveri example |
|---|---|---|
| **Document** | One piece of text you want to be findable | One help-centre article |
| **Query** | What the user types | "my hospital bill was not paid" |
| **Embedding** | A list of numbers that stands for the *meaning* of a piece of text | `[0.03, -0.21, 0.88, ...]` — 384 numbers |
| **Vector** | The mathematician's word for "a list of numbers" | Same thing as above |

**"Embedding" and "vector" are used interchangeably** in this field. An embedding *is* a vector. When someone says "vector database", they mean "a database of embeddings".

---

### 🧠 The one-sentence version of this entire session

> **We turn every piece of text into a point in space. Text that means similar things lands in similar places. Search then becomes: "which points are nearest to the user's point?"**

That is it. Everything else today is detail.

---

### 🗺️ Where today sits in the bigger picture

```
                    ┌──────────────────────────────────┐
   Day 9 (today)    │  Text  ──►  Numbers (embeddings) │
                    │  How do we measure "similar"?    │
                    └───────────────┬──────────────────┘
                                    │
                    ┌───────────────▼──────────────────┐
   Day 10 (next)    │  Store them · Search them fast   │
                    │  ChromaDB · FAISS · real code    │
                    └───────────────┬──────────────────┘
                                    │
                    ┌───────────────▼──────────────────┐
   Later            │  Feed results to Claude  =  RAG  │
                    └──────────────────────────────────┘
```

Embeddings are the foundation of **RAG, AI search, recommendations, deduplication, and AI memory**. Get today right and the rest of the program is much easier.

---

# Section 1 — Why keyword search breaks

## 🧠 The idea

Old-style search does one thing: **it counts words that appear in both the query and the document.**

That is all. It has no idea what any word *means*. To a keyword engine, "dog" and "puppy" are as unrelated as "dog" and "asteroid" — they are simply different strings of letters.

So keyword search fails in four predictable ways:

| Failure | Example at Kaveri | Why it breaks |
|---|---|---|
| **Synonyms** | "bill not paid" vs "claim declined" | Different words, same meaning |
| **Jargon gap** | Customer says "hospital bill", company says "cashless request" | Users don't know your internal vocabulary |
| **Paraphrase** | "how long till I get my money" vs "reimbursement processing timeline" | Same question, zero shared words |
| **Common words win** | "the policy for the claim" | "the" appears everywhere and matches everything |

## 🔬 How it actually works (and why that's the problem)

A keyword engine treats a document as a **bag of words** — a bucket of loose word tokens with the order thrown away.

```
"my hospital bill was not paid"  ──►  {my, hospital, bill, was, not, paid}
"Cashless request declined"      ──►  {cashless, request, declined}

Overlap = {}   ──►  score 0   ──►  never shown to the customer
```

Notice something alarming: the bag also throws away **"not"** as just another word. To a bag-of-words engine, *"claim was paid"* and *"claim was not paid"* look almost identical. Word counting has no concept of meaning at all.

> ⚠️ **Accuracy note:** real keyword engines (Elasticsearch, Lucene, Postgres full-text) are much smarter than raw word counting. They use **BM25**, which down-weights common words like "the" and rewards rare words, plus stemming ("running" → "run") and synonym lists. That fixes *some* of the above. What BM25 still **cannot** do is connect "hospital bill" to "cashless request" — because it has no model of meaning, only of word statistics. This distinction matters in interviews: the problem with keyword search is not that it is dumb, it is that **it is a statistics engine, not a meaning engine**.

## 💻 Run it — watch keyword search fail

This cell is plain Python. No AI, no downloads. It shows the failure in 20 lines.

In [3]:
# Kaveri Insurance help centre - 6 articles.
articles = [
    "Cashless request declined - how to claim reimbursement",
    "Hospital network list for Tamil Nadu",
    "Documents needed to file a medical claim",
    "How to pay your annual premium online",
    "Why your claim was rejected and what to do next",
    "Adding a family member to your policy",
]

query = "my hospital bill was not paid"

# Keyword search = count the words the query and the article share.
query_words = set(query.lower().split())

print(f'Customer typed: "{query}"')
print(f"Query words   : {sorted(query_words)}\n")

for article in articles:
    article_words = set(article.lower().replace("-", " ").split())
    shared = query_words & article_words
    print(f"  score {len(shared)}  | {article}")

print("\n--- What went wrong ---")
print("The BEST article ('Cashless request declined') scored 0.")
print("The two articles that scored 1 did so on 'hospital' and on 'was' -")
print("one is off-topic, the other matched purely on a filler word.")
print("Keyword search matched LETTERS. The customer asked about MEANING.")

Customer typed: "my hospital bill was not paid"
Query words   : ['bill', 'hospital', 'my', 'not', 'paid', 'was']

  score 0  | Cashless request declined - how to claim reimbursement
  score 1  | Hospital network list for Tamil Nadu
  score 0  | Documents needed to file a medical claim
  score 0  | How to pay your annual premium online
  score 1  | Why your claim was rejected and what to do next
  score 0  | Adding a family member to your policy

--- What went wrong ---
The BEST article ('Cashless request declined') scored 0.
The two articles that scored 1 did so on 'hospital' and on 'was' -
one is off-topic, the other matched purely on a filler word.
Keyword search matched LETTERS. The customer asked about MEANING.


### ✅ Quick check

<details>
<summary><b>Q1. Which article should have been rank 1, and what score did it get?</b> (click)</summary>

"Cashless request declined — how to claim reimbursement" should be rank 1. It scored **0**, because it shares zero words with the query. This is the whole problem in one line.
</details>

<details>
<summary><b>Q2. The customer's query and the "Hospital network list" article share the word "hospital". Is that article useful to this customer?</b> (click)</summary>

No. The customer wants their money. The network list tells them which hospitals are covered. Keyword search ranked a **useless but word-matching** article above a **useful but word-different** one. Keyword search optimises for the wrong thing.
</details>

### 🎤 Interview angle

> **Q: "Why do companies still use keyword search if it is this bad?"**
>
> **A:** "Because it is not bad at everything — it is excellent at exact matching. If a user searches an order ID, an error code, a product SKU, or a person's name, keyword search gets it exactly right and semantic search often gets it *wrong* by returning something that merely looks similar. The right production answer is usually **hybrid search**: run both, and blend the scores. Semantic search handles 'my bill wasn't paid'; keyword search handles 'policy KVI-2024-88123'."

### 🤯 Fun fact

**BM25**, still the default ranking function in Elasticsearch and Lucene today, comes from the **Okapi** information-retrieval system built at City University London in the 1980s and 90s — the "BM" stands for **Best Matching**, and it is roughly the 25th formula the researchers tried. A formula from before the web still powers a large share of enterprise search.

---

# Section 2 — What is an embedding?

## 🧠 The idea (the map analogy — use this one in interviews)

Think about a map of India.

Every city is written down as **two numbers**: latitude and longitude.

```
Chennai   →  (13.08, 80.27)
Bengaluru →  (12.97, 77.59)
Delhi     →  (28.61, 77.21)
```

Now answer this without looking at a map: **is Chennai closer to Bengaluru or to Delhi?**

You can answer it with arithmetic alone, because the numbers **carry real information about position**. Cities that are near each other got similar numbers. That is not a coincidence — that is the entire design of latitude and longitude.

**An embedding does exactly this for meaning.**

```
"claim declined"      →  (0.83, 0.11, -0.42, ... )   384 numbers
"cashless rejected"   →  (0.81, 0.09, -0.39, ... )   ← very close by
"annual premium due"  →  (-0.22, 0.71, 0.05, ... )   ← far away
```

> **An embedding is a coordinate for meaning.** Latitude and longitude place a city on Earth. An embedding places a *sentence* in "meaning space". Sentences that mean similar things get similar coordinates — even when they share no words at all.

That single sentence is the answer to "what is an embedding?" in any interview, at any level.

## 🔬 How it actually works — text in, numbers out

Here is the real pipeline inside the model. Four steps.

```
  "my hospital bill was not paid"
              │
              ▼
  ① TOKENISE - split into word pieces the model knows
     ["my", "hospital", "bill", "was", "not", "paid"]
              │
              ▼
  ② ENCODE - a Transformer reads ALL the words together,
     so each word's vector is coloured by its neighbours.
     "bill" next to "hospital" != "bill" next to "parliament".
     Output: one vector per word piece.
              │
              ▼
  ③ POOL - average all the word vectors into ONE vector.
     (This is called mean pooling.)
              │
              ▼
  ④ NORMALISE - scale it so its length is exactly 1.
              │
              ▼
  [0.031, -0.208, 0.884, ... ]   <- 384 numbers. One sentence. Done.
```

**The two things people miss:**

- **Step ②** is why embeddings beat keyword search. The model reads the *whole* sentence before deciding what each word means. This is called *contextual* encoding, and it is what a Transformer is for.
- **Step ③** is why a 5-word query and a 500-word document produce the **same size** vector. Length of input does not change length of output. That is what makes comparison fast and uniform.

> ⚠️ **Accuracy note #1 — the one everyone gets wrong.** You will read explanations saying "dimension 1 means *is it about animals*, dimension 2 means *is it positive*". **This is false.** Individual dimensions of a real embedding are **not human-readable** and do not correspond to named concepts. Meaning is spread across all 384 numbers together — no single number "means" anything on its own. Use the map analogy (*coordinates*) for intuition, never the "each dimension is a labelled feature" story. Saying that in an interview signals you learned this from a blog post rather than from the model.

> ⚠️ **Accuracy note #2.** "Embeddings understand meaning" is a shortcut. Precisely: the model was **trained so that texts humans treat as related end up near each other**. It learned a statistical map of usage, not comprehension. That is why it inherits biases from its training data, and why it can fail badly on vocabulary it never saw — like your company's internal product codes.

## 💻 Run it — see the numbers appear

In [4]:
from sentence_transformers import SentenceTransformer

# all-MiniLM-L6-v2 is small (22.7M parameters) and fast. First run downloads ~90 MB.
model = SentenceTransformer("all-MiniLM-L6-v2")

texts = [
    "my hospital bill was not paid",
    "cashless request declined",
    "how to pay my annual premium online",
]

vectors = model.encode(texts)

print(f"Input : {len(texts)} sentences")
print(f"Output: array of shape {vectors.shape}   <- ({len(texts)} sentences, 384 numbers each)\n")

for text, vec in zip(texts, vectors):
    preview = ", ".join(f"{v:+.3f}" for v in vec[:6])
    print(f'"{text}"')
    print(f"   -> [{preview}, ... ] ({len(vec)} numbers total)\n")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Input : 3 sentences
Output: array of shape (3, 384)   <- (3 sentences, 384 numbers each)

"my hospital bill was not paid"
   -> [+0.012, +0.075, +0.028, -0.023, -0.011, -0.044, ... ] (384 numbers total)

"cashless request declined"
   -> [-0.059, +0.102, -0.023, -0.019, -0.024, -0.080, ... ] (384 numbers total)

"how to pay my annual premium online"
   -> [-0.019, +0.013, -0.012, -0.025, +0.018, +0.043, ... ] (384 numbers total)



**What you just saw:** three sentences of very different lengths became three lists of **exactly 384 numbers**. Those numbers are the coordinates. In the next section we measure the distance between them.

### 💻 Run it — the length of the text does not change the size of the vector

This surprises everyone the first time.

In [5]:
short = "claim"
long_text = (
    "The policyholder submitted a cashless authorisation request at a network "
    "hospital in Chennai. The request was declined by the third party administrator "
    "due to incomplete pre-authorisation documents, and the policyholder was advised "
    "to settle the bill directly and file for reimbursement within thirty days."
)

for name, text in [("1 word", short), ("55 words", long_text)]:
    v = model.encode(text)
    magnitude = float((v ** 2).sum() ** 0.5)
    print(f"{name:9s} -> vector length: {len(v)}   magnitude: {magnitude:.4f}")

print("\nBoth give 384 numbers. Both have magnitude 1.0 (unit length).")
print("This is why comparing a short query to a long document is fast and fair.")

1 word    -> vector length: 384   magnitude: 1.0000
55 words  -> vector length: 384   magnitude: 1.0000

Both give 384 numbers. Both have magnitude 1.0 (unit length).
This is why comparing a short query to a long document is fast and fair.


> **Why magnitude 1.0?** `all-MiniLM-L6-v2` and `all-mpnet-base-v2` both end with a normalisation step, so every output vector has length exactly 1. This is not a detail to skip — it is what makes the cosine maths in Section 3 simple, and it is what lets FAISS use fast dot-product search. We come back to it.

## 🏢 In the real world

**Spotify** uses embeddings of listening behaviour to power Discover Weekly — songs played in similar contexts land near each other, so you get recommendations from artists you have never heard of.

**Kaveri Insurance (our scenario)** will use text embeddings so that "my hospital bill was not paid" lands next to "cashless request declined" — which is what we build tomorrow.

**Anthropic** takes a clear position here, and it is worth knowing for interviews: **Anthropic does not sell its own embedding model.** The official docs point developers to **Voyage AI** (models `voyage-4`, `voyage-4-lite`, `voyage-4-nano`; 32,000-token context; 1024 dimensions by default). This tells you something architectural: **embedding models and chat models are different tools for different jobs.** You use a small embedding model to *find* the right 5 documents out of a million, and then you use Claude to *reason* over those 5. Using Claude to read all million would cost a fortune and take hours.

## ⚠️ Don't mix these up

> ❌ **Wrong:** "An embedding is a compressed version of the text — you can decompress it back."
> ✅ **Right:** An embedding is **one-way**. You cannot get the original sentence back from the vector. It keeps meaning, not words.

> ❌ **Wrong:** "Bigger vectors are always better."
> ✅ **Right:** 768 dimensions (MPNet) beats 384 (MiniLM) on quality benchmarks, but costs 2× the memory and is several times slower to encode. For most enterprise search, MiniLM is the correct engineering choice. Bigger is a *tradeoff*, not an upgrade.

> ❌ **Wrong:** "Embeddings are the same as tokens."
> ✅ **Right:** A **token** is a piece of a word (input). An **embedding** is the meaning-vector for the whole text (output). Many tokens go in, one embedding comes out.

> ❌ **Wrong:** "Claude turns my text into embeddings."
> ✅ **Right:** Claude is a chat/reasoning model. Anthropic's docs state plainly that Anthropic does not offer an embedding model, and recommend Voyage AI. In this course we generate embeddings locally with MiniLM/MPNet, and use **Claude for the reasoning step on top**.

## 🎤 Interview angle

> **Q (beginner): "Explain embeddings to my grandmother."**
>
> **A:** "Every city has a latitude and longitude, and cities that are close together have similar numbers — so you can work out which cities are near each other using only arithmetic. An embedding does the same for sentences. It gives every sentence a set of coordinates, and sentences that mean similar things get similar coordinates. Search becomes 'find the nearest coordinates'."

> **Q (architecture): "Why not just send all my documents to Claude and ask it to find the right one?"**
>
> **A:** "Cost, latency, and context limits. Embedding a million documents is a one-time, cheap, local computation, and searching them takes milliseconds. Sending a million documents through any LLM on every query costs orders of magnitude more and will not fit in a context window. The standard architecture is **retrieve with embeddings, reason with Claude** — cheap narrowing first, expensive intelligence last."

### ✅ Quick check

<details>
<summary><b>Q1. True or False: you can reconstruct the original sentence from its embedding.</b></summary>

**False.** Embedding is one-way. It preserves meaning, not the exact words. (Note for the security-minded: research has shown embeddings can leak *approximate* content, so treat stored vectors as sensitive data, not as anonymised data.)
</details>

<details>
<summary><b>Q2. A 3-word query and a 900-word document. How many numbers in each embedding?</b></summary>

**The same number** — 384 with MiniLM, 768 with MPNet. Input length does not affect output size. That is the point of the pooling step.
</details>

<details>
<summary><b>Q3. Your CTO says "let's use 4096-dimensional embeddings, more is better." What do you say?</b></summary>

"More dimensions usually improve quality a little, but memory and search cost scale linearly with dimensions. At 10 million documents, 384-d costs about 15 GB of float32 vectors and 4096-d costs about 160 GB. Let's benchmark retrieval accuracy on our own queries at 384, 768, and 1024 first — the quality gain is often smaller than the bill."
</details>

### 🤯 Fun fact

`all-MiniLM-L6-v2` was trained during a **Hugging Face community sprint in 2021** on donated Google TPUs (7 × TPU v3-8), on **1,170,060,424 sentence pairs** drawn from Reddit, Stack Exchange, Wikipedia, and academic papers. The model is only **22.7 million parameters** — a rounding error next to a frontier LLM — and it is downloaded roughly **250 million times a month** from Hugging Face. A tiny model built at a community hackathon is one of the most-used AI models on Earth.

---

# Section 3 — Which embedding model? MiniLM vs MPNet

## 🧠 The idea

You do not build an embedding model. You **choose** one and download it. There are thousands on Hugging Face. For English text, two are the workhorses of the industry:

| | **all-MiniLM-L6-v2** | **all-mpnet-base-v2** |
|---|---|---|
| Numbers per text (dimensions) | **384** | **768** |
| Size (parameters) | 22.7 million | ~110 million |
| Download size | ~90 MB | ~420 MB |
| Layers | 6 | 12 |
| Max input before it **silently truncates** | **256** word pieces | **384** word pieces |
| Trained on | ~1.17 billion sentence pairs | ~1 billion sentence pairs |
| Base model it was built from | `nreimers/MiniLM-L6-H384-uncased` | `microsoft/mpnet-base` |
| Speed | Fast | Several times slower |
| Quality | Very good | Better |
| **Use it when** | You have lots of documents, need speed, or run on CPU | Quality matters more than latency; documents are longer |

**Both** are trained the same way, by the same project, and **both output unit-length vectors**. Swapping between them in code is a one-line change — which is exactly why you should benchmark both on your own data instead of arguing about it.

> ⚠️ **Accuracy note — the silent truncation trap.** That "max input" row is the single most expensive line in this table. If you feed a 2,000-word policy document to MiniLM, it **does not error**. It quietly reads the first ~256 word pieces (roughly 200 words) and throws the rest away. Your vector then represents only the opening paragraph. Teams lose weeks to this: "why does search miss content that is definitely in the document?" — because the model never saw it. **The fix is chunking**: split long documents into ~200-word pieces and embed each piece separately. We do this properly in Day 10.

## 🔬 How it actually works — what "L6" and "base" mean

The names are not random. Decode them:

```
all - MiniLM - L6 - v2
 │      │      │    └── version 2
 │      │      └─────── 6 Transformer layers (small = fast)
 │      └────────────── the MiniLM architecture (a distilled, shrunk BERT)
 └───────────────────── trained on "all" the datasets in the project

all - mpnet - base - v2
 │      │       │     └── version 2
 │      │       └──────── "base" size: 12 layers (the standard BERT size)
 │      └──────────────── the MPNet architecture from Microsoft Research
 └─────────────────────── same "all" training mixture
```

**What makes MPNet different from plain BERT?** BERT learns by hiding random words and guessing them, but it looks at each hidden word independently. MPNet, from Microsoft Research, combines that with predicting words in a shuffled order, so it also learns how hidden words depend on *each other*, while still seeing full sentence position information. In practice that buys a few points of accuracy on sentence-similarity benchmarks. **You do not need this detail to use the model** — you need it only if an interviewer asks "why MPNet and not BERT?"

## 💻 Run it — compare the two models on the Kaveri problem

In [6]:
from sentence_transformers import SentenceTransformer
import time

pair = ["my hospital bill was not paid", "cashless request declined"]

for name in ["all-MiniLM-L6-v2", "all-mpnet-base-v2"]:
    m = SentenceTransformer(name)

    start = time.perf_counter()
    vecs = m.encode(pair)
    elapsed_ms = (time.perf_counter() - start) * 1000

    # Both models output unit-length vectors, so the dot product IS the cosine similarity.
    similarity = float(vecs[0] @ vecs[1])

    print(f"{name:22s} dims={vecs.shape[1]:4d}  encode={elapsed_ms:6.1f} ms  similarity={similarity:.3f}")

print("\nBoth models see that these two sentences mean the same thing,")
print("even though they share ZERO words. That is semantic search working.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

all-MiniLM-L6-v2       dims= 384  encode=  17.1 ms  similarity=0.239


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

all-mpnet-base-v2      dims= 768  encode= 110.6 ms  similarity=0.289

Both models see that these two sentences mean the same thing,
even though they share ZERO words. That is semantic search working.


### 💻 Run it — prove the truncation trap is real

Do not take our word for it. Watch a model ignore the end of a document.

In [7]:
mini = SentenceTransformer("all-MiniLM-L6-v2")

# A long document where the ONLY useful sentence is buried at the very end.
filler = "This policy document describes general terms and conditions. " * 60
doc_long = filler + " IMPORTANT: dental treatment is excluded from this policy."
doc_short = "IMPORTANT: dental treatment is excluded from this policy."

query_vec = mini.encode("is dental covered?")

for label, doc in [("short doc (fits)", doc_short), ("long doc (truncated)", doc_long)]:
    score = float(mini.encode(doc) @ query_vec)
    words = len(doc.split())
    print(f"{label:22s} {words:4d} words  ->  similarity {score:.3f}")

print("\nSame key sentence, very different scores.")
print("In the long document the model never reached the dental sentence -")
print("it stopped after ~256 word pieces. THIS is why we chunk documents.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

short doc (fits)          8 words  ->  similarity 0.638
long doc (truncated)    488 words  ->  similarity 0.129

Same key sentence, very different scores.
In the long document the model never reached the dental sentence -
it stopped after ~256 word pieces. THIS is why we chunk documents.


## 🏢 In the real world

A mid-size Indian bank running semantic search over 2 million support tickets on CPU-only servers will pick **MiniLM**: 384 dimensions × 2M documents × 4 bytes = about **3 GB** of vectors, which fits comfortably in RAM. The same corpus in MPNet is **6 GB** and takes several times longer to index. If measured search quality is within a couple of percent, the architect picks MiniLM and spends the saved budget on chunking and reranking — which usually improve results more than the bigger model does.

## 🎤 Interview angle

> **Q (intermediate): "How do you choose an embedding model?"**
>
> **A:** "I start from constraints, not from leaderboards. Three questions: What languages? How many documents and what latency budget? How long is a typical document? Then I build a small evaluation set — 30 to 50 real user queries with the documents that *should* be returned — and measure recall@5 for two or three candidate models on my own data. The MTEB leaderboard is a starting shortlist, not an answer, because my domain is not their benchmark."

> **Q (advanced): "You switched embedding models in production. What breaks?"**
>
> **A:** "Everything already in the index. Vectors from two different models are not comparable — they live in different spaces, and MiniLM's 384 dimensions will not even load into a 768-dimension index. Changing the embedding model means a **full re-index of the entire corpus**. So the model choice is a long-lived architectural decision, and I would plan for a dual-write or blue/green index switchover, not an in-place change."

### ✅ Quick check

<details>
<summary><b>Q1. You embed a 50-page PDF with MiniLM as one string. What is in the resulting vector?</b></summary>

Roughly the **first 200 words**. Everything after ~256 word pieces was silently discarded. The vector is a fair representation of page 1 and a completely wrong representation of the document. Chunk it.
</details>

<details>
<summary><b>Q2. Can you search a MiniLM query vector against an index built with MPNet?</b></summary>

**No.** Different dimensions (384 vs 768) so it will not even fit, and even if the sizes matched, the two models place meaning in different coordinate systems. One model for the whole index — always.
</details>

### 🤯 Fun fact

MiniLM and MPNet come from **rival labs**. MiniLM came out of **Microsoft Research** in 2020 as a *distillation* experiment — teaching a small model to imitate a big one. MPNet also came out of **Microsoft Research**, in the same year, as a *pretraining* experiment combining BERT's and XLNet's training tricks. The versions everyone actually uses (`all-MiniLM-L6-v2`, `all-mpnet-base-v2`) were then fine-tuned by neither of them, but by the **open-source Sentence-Transformers community** during a Hugging Face sprint. Modern AI infrastructure is very often a community fine-tune of a corporate research artefact.

---

# Section 4 — Cosine similarity in plain terms

## 🧠 The idea

We have two lists of 384 numbers. **How do we turn that into one score that says "these mean the same thing"?**

The trick is to stop thinking about *lists* and start thinking about **arrows**.

A vector is an arrow from the origin (0,0) pointing to a place. Two arrows can differ in two ways:

1. **Direction** — which way they point
2. **Length** — how far they reach

**Cosine similarity looks only at direction and completely ignores length.**

```
          ↑
        B │    ,A            A and B point the SAME way.
          │  ,'              Different lengths, same direction.
          │,'                Cosine similarity = 1.0
   ───────┼────────→
          │
```

Why ignore length? Because **length carries no meaning here**. A long document is not "more meaningful" than a short one — it is just longer. Direction is where the meaning lives.

## 🔬 How it actually works — the formula, in three plain steps

$$\text{cosine similarity}(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|}$$

In words:

| Symbol | Plain English | How to compute it |
|---|---|---|
| $A \cdot B$ | **Dot product**: multiply the two lists position by position, add it all up | `sum(a*b for a,b in zip(A,B))` |
| $\|A\|$ | **Magnitude**: how long the arrow is | `sqrt(sum(a*a for a in A))` |
| the division | Cancels out length, leaving only direction | — |

**Worked example you can do on paper.** Let `A = [3, 4]` and `B = [6, 8]`.

```
Dot product   A·B = (3×6) + (4×8) = 18 + 32 = 50
Magnitude    |A|  = sqrt(3² + 4²) = sqrt(25)  = 5
Magnitude    |B|  = sqrt(6² + 8²) = sqrt(100) = 10

cosine = 50 / (5 × 10) = 50 / 50 = 1.0     ← identical direction ✓
```

B is literally A doubled. Straight-line (Euclidean) distance between them is 5 — which would call them "far apart". Cosine correctly calls them **identical**. That is the whole argument for cosine over raw distance.

## 💻 Run it — cosine similarity from scratch, no libraries

In [8]:
import math

def dot(a, b):
    return sum(x * y for x, y in zip(a, b))

def magnitude(a):
    return math.sqrt(sum(x * x for x in a))

def cosine_similarity(a, b):
    return dot(a, b) / (magnitude(a) * magnitude(b))

A = [3, 4]     # our reference arrow
B = [6, 8]     # same direction, twice as long
C = [4, 3]     # slightly different direction
D = [-3, -4]   # exact opposite direction

for name, v in [("B (A doubled)", B), ("C (tilted)", C), ("D (reversed)", D)]:
    cos = cosine_similarity(A, v)
    euclid = math.dist(A, v)
    angle = math.degrees(math.acos(max(-1.0, min(1.0, cos))))
    print(f"A vs {name:14s} cosine={cos:+.3f}  angle={angle:5.1f} deg  euclidean={euclid:.2f}")

print("\nRead the B row carefully:")
print("  cosine says 'identical' (1.000) but euclidean says 'far apart' (5.00).")
print("  For meaning, cosine is the one telling the truth.")

A vs B (A doubled)  cosine=+1.000  angle=  0.0 deg  euclidean=5.00
A vs C (tilted)     cosine=+0.960  angle= 16.3 deg  euclidean=1.41
A vs D (reversed)   cosine=-1.000  angle=180.0 deg  euclidean=10.00

Read the B row carefully:
  cosine says 'identical' (1.000) but euclidean says 'far apart' (5.00).
  For meaning, cosine is the one telling the truth.


### 💻 Run it — cosine similarity on real sentences

In [9]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer("all-MiniLM-L6-v2")

sentences = [
    "my hospital bill was not paid",       # 0 - the customer
    "cashless request declined",           # 1 - same meaning, no shared words
    "claim rejected by the insurer",       # 2 - same topic
    "how to pay my annual premium",        # 3 - different topic, same domain
    "best street food in Chennai",         # 4 - totally unrelated
]

vecs = model.encode(sentences)
scores = vecs @ vecs.T          # unit vectors -> dot product IS cosine similarity

print("Similarity of sentence 0 to every other sentence:\n")
for i, s in enumerate(sentences):
    marker = "  <- the query" if i == 0 else ""
    print(f"  {scores[0][i]:+.3f}   \"{s}\"{marker}")

print("\nNotice the ORDER is exactly right:")
print("  declined > rejected > premium > street food")
print("Sentence 1 shares ZERO words with the query and still ranks first.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Similarity of sentence 0 to every other sentence:

  +1.000   "my hospital bill was not paid"  <- the query
  +0.239   "cashless request declined"
  +0.348   "claim rejected by the insurer"
  +0.416   "how to pay my annual premium"
  -0.025   "best street food in Chennai"

Notice the ORDER is exactly right:
  declined > rejected > premium > street food
Sentence 1 shares ZERO words with the query and still ranks first.


## ⚠️ Don't mix these up — three traps that end interviews

> ❌ **Wrong:** "0.0 means unrelated and -1.0 means opposite, so I'll treat anything below 0 as an antonym."
> ✅ **Right:** With modern sentence embedding models, real text **almost never scores below 0**. Unrelated sentences typically land around **0.0 to 0.3**, not at -1. The theoretical range is -1 to +1; the *practical* range you will actually observe is roughly 0 to 1. Any threshold you set must be measured on your own data and your own model.

> ❌ **Wrong:** "High cosine similarity means the two sentences agree."
> ✅ **Right:** **Cosine similarity measures topic, not agreement.** "I love this policy" and "I hate this policy" score **very high** together, because they are about the same thing. This is the single most common production surprise in semantic search — it will happily retrieve a document that says the exact opposite of what the user needs. Sentiment and negation need a separate step.

> ❌ **Wrong:** "0.8 is the industry-standard cutoff for a good match."
> ✅ **Right:** There is no universal cutoff. Thresholds are **model-specific and domain-specific**. MiniLM and MPNet produce differently-spread scores on the same pair. Always calibrate: take 30 real queries, look at the scores of results you know are good and results you know are bad, and put the line between them.

> ⚠️ **Accuracy note — cosine vs L2 distance, settled.** People argue about this constantly. Here is the fact: **if your vectors are unit length** (MiniLM's and MPNet's are), then squared Euclidean distance and cosine similarity are related by the exact identity
>
> $$\|A - B\|^2 = 2 - 2\cos(A, B)$$
>
> which means **they always rank results in the same order**. `IndexFlatL2` and cosine search return identical top-k on normalised vectors. The choice only changes the *number* you show the user, not the *ranking*. Where it genuinely matters is if your vectors are **not** normalised — then L2 is polluted by document length and cosine is not. Prove the identity yourself in the next cell.

## 💻 Run it — prove the L2 / cosine identity

In [10]:
import numpy as np

a, b = vecs[0], vecs[1]     # two real unit-length embeddings from the cell above

cos = float(a @ b)
l2_squared = float(((a - b) ** 2).sum())

print(f"cosine similarity        : {cos:.6f}")
print(f"squared L2 distance      : {l2_squared:.6f}")
print(f"2 - 2 * cosine           : {2 - 2 * cos:.6f}   <- identical")
print("\nSo on unit-length vectors, L2 and cosine rank results identically.")
print("Choose either. Just never mix the two scales in one system.")

cosine similarity        : 0.238744
squared L2 distance      : 1.522511
2 - 2 * cosine           : 1.522511   <- identical

So on unit-length vectors, L2 and cosine rank results identically.
Choose either. Just never mix the two scales in one system.


## 🎤 Interview angle

> **Q (beginner): "What is cosine similarity?"**
>
> **A:** "It is the angle between two vectors, expressed as a number from -1 to 1. Same direction is 1, perpendicular is 0, opposite is -1. We use it for text because it ignores vector length — a long document is not more meaningful than a short one, so we only care which way the vector points."

> **Q (advanced): "Your semantic search returns a document that says the exact opposite of what the user asked. Why, and how do you fix it?"**
>
> **A:** "Because cosine similarity measures topical closeness, not agreement — 'covered' and 'not covered' are about the same topic and sit close together. Three fixes, in the order I would try them: first, a **cross-encoder reranker** on the top 20 results, which reads query and document together and is much better at negation. Second, **metadata filters** so the retrieval is constrained structurally, not just semantically. Third, put the top results into **Claude** and let it decide which actually answers the question — an LLM handles negation well, and at 5 documents the cost is trivial."

### ✅ Quick check

<details>
<summary><b>Q1. Two vectors score 0.94. Same topic?</b></summary>

Almost certainly the same topic — but **not necessarily the same position on it**. 0.94 could be "the claim was approved" and "the claim was not approved". High similarity means "about the same thing", not "says the same thing".
</details>

<details>
<summary><b>Q2. Why divide by the magnitudes at all?</b></summary>

To cancel out length so only direction remains. Without dividing, you have the raw dot product, which grows when vectors are longer — so long documents would win regardless of relevance.
</details>

<details>
<summary><b>Q3. You are told your embeddings are already normalised. Which is cheaper: cosine or dot product?</b></summary>

**Dot product** — it is the same answer with the division skipped. This is why FAISS's `IndexFlatIP` (inner product) is the standard choice for normalised embeddings, and why Voyage AI's docs point out that for their vectors, dot product and cosine are identical.
</details>

### 🤯 Fun fact

Cosine similarity is not from AI at all. It comes from **Gerard Salton's SMART system at Cornell in the 1960s and 70s** — a text retrieval system built on punch-card-era hardware. The formula that ranks your embedding search today was designed to search library card catalogues, decades before neural networks worked. Meanwhile, the "meaning as coordinates" idea got its most famous demonstration in **2013**, when Google's **word2vec** showed that `king − man + woman` lands nearest to `queen`. (⚠️ Accuracy note even on the fun fact: that result only holds because the search *excludes the three input words* — without that exclusion the nearest vector is usually just `king` again. The regularity is real but weaker than the legend.)

---

# Section 5 — Vector databases in plain terms

## 🧠 The idea

You now have a million vectors. Where do you put them?

The honest first answer: **in a NumPy array**. For a small corpus that genuinely works, and any architect who tells you otherwise is selling something.

So what does a vector database actually add? Five things, and only the first one is about speed:

| What a plain array gives you | What a vector database adds |
|---|---|
| Compare against **every** vector, every time | **Approximate search** — skip 99% of vectors, keep almost all the accuracy |
| Gone when the process ends | **Persistence** — survives restarts, backups, deploys |
| Vectors only | **Documents + metadata stored together**, so results come back readable |
| Search everything or nothing | **Filters** — "only claims documents, only 2026, only Tamil Nadu" |
| Rebuild the whole array to add one document | **Add, update, delete** individual records safely while serving traffic |

> **A vector database is a normal database whose "WHERE clause" is `nearest to this meaning`.** That is the one-line answer.

## 🔬 How it actually works — how "skip 99% of vectors" is possible

**The brute-force problem.** One query against 1 million 384-dimension vectors is 384 million multiply-and-add operations. Modern NumPy does that in a fraction of a second — fine for one user, hopeless for a thousand users a second.

**The idea that fixes it: don't look everywhere. Look in the right neighbourhood.**

This is called **ANN — Approximate Nearest Neighbour**.

```
  Finding a person in Chennai
  ────────────────────────────────────────────────────────────
  BRUTE FORCE (exact)          ANN (approximate)
  Knock on all 2 million       1. Which neighbourhood are they in?
  doors, one by one.           2. Knock on 2,000 doors there.
  Always correct.              3. Done, 1000x faster.
  Always slow.                 Occasionally you miss someone
                               who just moved to the next street.
```

The two ANN families you must be able to name:

| Family | The trick | Used by |
|---|---|---|
| **IVF** (Inverted File) | Cluster all vectors into groups up front. At query time, only search the nearest few groups. | FAISS |
| **HNSW** (Hierarchical Navigable Small World) | Build a multi-level graph of "who is near whom", then walk the graph from a coarse level down to a fine one. | ChromaDB, Qdrant, Weaviate, pgvector |

> ⚠️ **Accuracy note — the word "approximate" is a real cost, not marketing.** ANN can and does miss correct results. The measurement is **recall@k**: of the 10 truly-nearest documents, how many did the index actually return? You *tune* this. In FAISS, `nprobe` controls how many clusters to search: `nprobe=1` might be 150× faster with 88% recall; `nprobe=10` might be 30× faster with 100% recall. Those are real numbers you will reproduce in Day 10. **Never quote a vector database's speed without stating its recall** — a fast index with 60% recall is a broken search engine, and this is exactly the kind of nuance senior interviewers probe for.

> ⚠️ **Accuracy note — you probably don't need ANN yet.** Below roughly **100,000 documents**, exact brute-force search on a modern CPU takes single-digit milliseconds. Adding an approximate index at that scale buys you nothing and costs you accuracy plus a tuning problem. Knowing *when not to* reach for the fancy thing is what separates an architect from a tutorial follower.

## 🗺️ The landscape — and the distinction people get wrong

The most common confusion in this whole topic: **FAISS is not a database.**

```
  ┌─────────────────────────────────────────────────────────────┐
  │ LIBRARY - an index that lives in your process's RAM         │
  │ You handle storage, metadata, updates, scaling yourself.    │
  │                                                             │
  │   FAISS (Meta)          ScaNN (Google)     hnswlib          │
  └─────────────────────────────────────────────────────────────┘
  ┌─────────────────────────────────────────────────────────────┐
  │ EMBEDDED DATABASE - runs inside your app, saves to disk      │
  │ Documents + metadata + vectors + filters, one pip install.  │
  │                                                             │
  │   ChromaDB              LanceDB            sqlite-vec       │
  └─────────────────────────────────────────────────────────────┘
  ┌─────────────────────────────────────────────────────────────┐
  │ SERVER / MANAGED - a service you deploy or rent              │
  │ Replication, auth, autoscaling, multi-tenancy.              │
  │                                                             │
  │   Qdrant   Weaviate   Milvus   Pinecone   pgvector          │
  └─────────────────────────────────────────────────────────────┘
```

| Tool | What it really is | Pick it when |
|---|---|---|
| **FAISS** | A similarity-search **library** from Meta | You want maximum control and speed, and you already have somewhere to store the actual text |
| **ChromaDB** | An **embedded vector database** | Prototypes, small-to-mid production, RAG apps — you want documents, metadata and filters with no server |
| **pgvector** | A **Postgres extension** | You already run Postgres and want vectors next to your existing relational data — often the boring, correct enterprise answer |
| **Qdrant / Weaviate / Milvus** | Self-hostable **vector servers** | Millions to billions of vectors, multiple services querying, you need replication and RBAC |
| **Pinecone** | A **managed cloud service** | You want zero operations and can accept the cost and the data-residency conversation |

**The FDE version of this table:** when a client asks "which vector database should we use?", the answer is almost never a benchmark. It is "which one does your team already know how to operate at 3 a.m.?" A Postgres shop should start with pgvector. A team with no infra should start with Chroma. Nobody's search quality was ever saved by picking the trendier database — but plenty of projects have been sunk by adding an unfamiliar system to the on-call rotation.

## 💻 Run it — build a vector "database" in 15 lines, so you can see there is no magic

In [11]:
import numpy as np
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

# This IS a vector store. Documents in a list, vectors in an array.
docs = [
    "Cashless request declined - how to claim reimbursement",
    "Hospital network list for Tamil Nadu",
    "Documents needed to file a medical claim",
    "How to pay your annual premium online",
    "Why your claim was rejected and what to do next",
    "Adding a family member to your policy",
]
doc_vectors = model.encode(docs)          # shape (6, 384), unit length

def search(query, k=3):
    q = model.encode(query)               # 1. embed the query
    scores = doc_vectors @ q              # 2. dot product = cosine (unit vectors)
    top = np.argsort(scores)[::-1][:k]    # 3. sort, take the best k
    return [(float(scores[i]), docs[i]) for i in top]

for score, doc in search("my hospital bill was not paid"):
    print(f"  {score:.3f}  {doc}")

print("\nThat is semantic search. Three lines of real logic.")
print("A vector database adds persistence, filters, and approximate search -")
print("valuable, but it is not where the intelligence lives.")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

  0.376  Documents needed to file a medical claim
  0.375  Cashless request declined - how to claim reimbursement
  0.357  How to pay your annual premium online

That is semantic search. Three lines of real logic.
A vector database adds persistence, filters, and approximate search -
valuable, but it is not where the intelligence lives.


## 🏢 In the real world

**ChromaDB** was founded in **2022** by Jeff Huber and Anton Troynikov, released its open-source database on **14 February 2023**, and raised an **$18M seed round led by Quiet Capital**. Its core was later rewritten in **Rust** for performance — you can see this in the installed package, where the Python client talks to a Rust engine. It is the default choice in a large share of RAG tutorials and prototypes.

**FAISS** was released by **Meta (then Facebook) AI Research on 22 February 2017**. The accompanying paper, *Billion-scale similarity search with GPUs* by Johnson, Douze and Jégou, reported nearest-neighbour search roughly **8.5× faster than the previous state of the art**, and built the first k-nearest-neighbour graph over **1 billion** high-dimensional vectors. It remains the reference implementation that other vector databases are measured against.

## ⚠️ Don't mix these up

> ❌ **Wrong:** "A vector database makes search smarter."
> ✅ **Right:** The **embedding model** determines search quality. The **database** determines speed, scale and operations. Swapping Chroma for Pinecone will not fix bad results; swapping the embedding model or adding a reranker might.

> ❌ **Wrong:** "FAISS is a database."
> ✅ **Right:** FAISS is a **library**. It holds an index in memory and returns row numbers. It does not store your text, does not do metadata filters, and does not persist unless you explicitly save the index to a file.

> ❌ **Wrong:** "Vector databases are fast because they use GPUs."
> ✅ **Right:** They are fast because of **the index structure** (IVF, HNSW) — skipping work, not doing work faster. GPUs help, but the algorithm is the source of the 100×.

## 🎤 Interview angle

> **Q (architecture): "Design search for 50 million support tickets, 500 queries per second, sub-200 ms."**
>
> **A:** "Chunk tickets to about 200 words and embed with MiniLM at 384 dimensions — that is roughly 75 GB of float32 vectors before compression, so this does not fit one commodity machine comfortably. I would shard by tenant or by date across nodes, use HNSW for its stable low latency at high queries-per-second, and add product quantisation if memory is the binding constraint. Embedding happens in a batch pipeline offline; only the query is embedded at request time, and I cache popular query vectors. I would set my recall target first — say recall@10 of 95% — and tune the index to hit it, because latency numbers without a recall target are meaningless."

> **Q (FDE): "The client says 'we already have Elasticsearch, why do we need another database?'"**
>
> **A:** "You might not. Elasticsearch supports dense vector search, and pgvector does too if you are a Postgres shop. The question is not 'which database' but 'is your retrieval quality good enough'. I would run a 30-query bake-off on their real content — their existing keyword search versus semantic versus hybrid — and let the numbers decide. If keyword search already answers 90% of their queries, we add semantic as a fallback rather than replacing anything."

### ✅ Quick check

<details>
<summary><b>Q1. 40,000 documents, one internal tool, 20 users. Do you need a vector database?</b></summary>

**No.** 40,000 × 384 floats is about 60 MB — it fits in RAM, and brute-force search takes a few milliseconds. A NumPy array plus a pickle file is a completely respectable answer. Use Chroma if you want metadata filters and persistence for free, not because you need the speed.
</details>

<details>
<summary><b>Q2. Your search got faster after switching to an ANN index, but users complain results got worse. What happened?</b></summary>

You traded recall for speed and did not measure it. The index is skipping clusters that contained good results. Raise `nprobe` (FAISS) or `ef_search` (HNSW) until recall@10 is back where you need it, then re-measure latency.
</details>

<details>
<summary><b>Q3. What does a vector database store that a FAISS index does not?</b></summary>

The **documents themselves and their metadata**, plus the durable storage and filtering around them. FAISS gives you back an integer row number and a distance; you must look up what document that row was.
</details>

### 🤯 Fun fact

The **HNSW** algorithm that powers ChromaDB, Qdrant, Weaviate and pgvector is named after the **"small world"** phenomenon — the same six-degrees-of-separation idea from social networks. The insight is identical: in a network with mostly local links plus a few long-range shortcuts, you can reach any node in a handful of hops. HNSW builds that structure deliberately over your vectors, so a search "teleports" across the space in a few long jumps and then walks locally to the answer. A sociology observation from the 1960s became the default index in modern AI infrastructure.

---

# Architecture — the whole system on one page

This is the picture you should be able to draw on a whiteboard from memory. It is also exactly what you build tomorrow.

```
╔══════════════════════════════════════════════════════════════════════════╗
║  OFFLINE - runs once, then whenever content changes (a batch job)        ║
╚══════════════════════════════════════════════════════════════════════════╝

  Kaveri help centre        ┌──────────┐      ┌───────────────┐     ┌──────────────┐
  4,000 articles      ───►  │  CHUNK   │ ───► │  EMBED        │ ──► │ VECTOR STORE │
  (PDF, HTML, DB)           │ ~200 wds │      │ MiniLM 384-d  │     │ ChromaDB     │
                            └──────────┘      └───────────────┘     │ vec+text+meta│
                                 ▲                    ▲             └──────────────┘
                     truncation is silent -    one model for the           ▲
                     chunking is not optional  WHOLE index, forever        │
                                                                          │
╔═════════════════════════════════════════════════════════════════════════╪╗
║  ONLINE - runs on every user query, budget ~200 ms                      ││
╚═════════════════════════════════════════════════════════════════════════╪╝
                                                                          │
  Customer types                                                          │
  "my hospital bill    ┌──────────────┐    ┌──────────────┐               │
   was not paid"  ───► │ EMBED QUERY  │ ─► │  SEARCH      │ ◄─────────────┘
                       │ SAME model!  │    │ top-k = 5    │
                       │ ~10 ms       │    │ + filters    │
                       └──────────────┘    └──────┬───────┘
                                                  │ 5 articles
                                                  ▼
                                       ┌─────────────────────┐
                                       │  (optional) RERANK  │  cross-encoder
                                       │  20 -> 5, better    │  fixes negation
                                       └──────────┬──────────┘
                                                  ▼
                                       ┌─────────────────────┐
                                       │  CLAUDE HAIKU 4.5   │  reads ONLY the
                                       │  writes the answer  │  retrieved text,
                                       │  cites the article  │  cites its source
                                       └──────────┬──────────┘
                                                  ▼
                                            Answer to Priya's customer
```

### Who is responsible for what

| Component | Its one job | If it fails |
|---|---|---|
| **Chunker** | Cut documents small enough that nothing is silently truncated | Content exists but is invisible to search — the worst kind of bug, because nothing errors |
| **Embedding model** | Turn text into meaning-coordinates | Bad or wrong-language model ⇒ irrelevant results no database can fix |
| **Vector store** | Store and return the nearest k, fast, with filters | Slow queries, or (with ANN) quietly missing correct results |
| **Reranker** *(optional)* | Re-order the top 20 by reading query+document together | Subtle relevance loss; the system still works without it |
| **Claude** | Read the retrieved text and write a grounded, cited answer | Hallucination risk if you let it answer without retrieved context |

### The three failure points that actually bite in production

1. **Model drift between index and query.** The index was built with MPNet, someone deploys a query path using MiniLM. No error is raised — results just turn to noise. **Guard:** store the embedding model name and dimension as metadata on the collection and assert it at startup.
2. **Silent truncation.** Covered above. **Guard:** log token counts at ingestion and alert on any chunk that hits the model's limit.
3. **Retrieval succeeded, answer still wrong.** The right document was retrieved but Claude was given a weak instruction and answered from its own knowledge. **Guard:** instruct Claude to answer *only* from the provided context and to say "not in the documents" otherwise — then evaluate that behaviour deliberately.

### Cost, security, scale — the architect's checklist

- **Cost.** Embedding is a one-time cost per document and it is tiny — MiniLM on CPU handles thousands of documents per minute for free. The recurring cost is the **LLM call at the end**. Claude Haiku is the cheapest Claude model and the right default here; keep retrieved context to the top 3–5 chunks rather than 20, because context length is what drives your bill.
- **Security.** Embeddings are **not anonymisation** — treat the vector store as holding the underlying sensitive data, with the same access controls. Apply tenant filters at the *database* level, never by filtering results in application code after the fact.
- **Scale.** Embedding is embarrassingly parallel — throw batch workers at it. Query latency is dominated by the index; that is where HNSW/IVF tuning earns its keep.
- **Observability.** Log every query, its top-k results with scores, and whether the user clicked anything. That log is the evaluation set you will wish you had in month three.

---

# 💻 Claude API Lab — putting embeddings and Claude together

**Objective:** show, with real API calls, why a production system needs **both** an embedding model and Claude — and why neither one alone is enough.

We run three parts:

| Part | What it proves |
|---|---|
| **A** | Embeddings retrieve the right articles cheaply |
| **B** | Claude turns retrieved articles into a real customer answer |
| **C** | Claude catches the negation trap that cosine similarity walks straight into |

---

### Part A — retrieve with embeddings

**Objective:** find the 3 most relevant Kaveri articles for a customer's message.

In [12]:
import numpy as np
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("all-MiniLM-L6-v2")

knowledge_base = [
    "Cashless request declined: if the hospital's cashless authorisation is rejected, "
    "pay the bill yourself and file for reimbursement within 30 days with the discharge "
    "summary, final bill, and payment receipt.",

    "Hospital network: Kaveri Insurance has 4,200 network hospitals across Tamil Nadu "
    "where cashless treatment is available. Check the network list before admission.",

    "Claim rejection reasons: the most common causes are missing pre-authorisation, "
    "treatment inside the 30-day waiting period, and non-disclosed pre-existing conditions.",

    "Premium payment: annual premiums can be paid online by UPI, net banking or card. "
    "A 15-day grace period applies after the due date.",

    "Adding a dependent: spouse and children can be added at renewal. Parents can be "
    "added mid-term with fresh medical underwriting.",
]

kb_vectors = embedder.encode(knowledge_base)

customer_message = "I was treated at a hospital in Chennai but my bill was not paid by insurance. What do I do?"

query_vector = embedder.encode(customer_message)
scores = kb_vectors @ query_vector
top_3 = np.argsort(scores)[::-1][:3]

retrieved = [knowledge_base[i] for i in top_3]

print(f'Customer: "{customer_message}"\n')
print("Retrieved by the embedding model:")
for rank, i in enumerate(top_3, 1):
    print(f"  {rank}. [{scores[i]:.3f}] {knowledge_base[i][:70]}...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Customer: "I was treated at a hospital in Chennai but my bill was not paid by insurance. What do I do?"

Retrieved by the embedding model:
  1. [0.496] Cashless request declined: if the hospital's cashless authorisation is...
  2. [0.451] Hospital network: Kaveri Insurance has 4,200 network hospitals across ...
  3. [0.372] Premium payment: annual premiums can be paid online by UPI, net bankin...


**Expected result:** the "Cashless request declined" article ranks first — even though the customer never used the words "cashless", "declined" or "reimbursement".

---

### Part B — reason with Claude

**Objective:** turn those 3 raw articles into a reply Priya could actually send.

Read the system prompt carefully. Two instructions in it are doing all the safety work: *answer only from the context*, and *say so if the answer is not there*.

In [13]:
context = "\n\n".join(f"[Article {i+1}] {doc}" for i, doc in enumerate(retrieved))

reply = client.messages.create(
    model=MODEL,
    max_tokens=400,
    system=(
        "You are a support agent at Kaveri Insurance. "
        "Answer ONLY using the provided articles. "
        "If the articles do not contain the answer, say so plainly. "
        "Cite the article number you used. Be warm and under 120 words."
    ),
    messages=[{
        "role": "user",
        "content": f"ARTICLES:\n{context}\n\nCUSTOMER MESSAGE:\n{customer_message}"
    }],
)

print(reply.content[0].text)
print("\n" + "-" * 60)
print(f"stop_reason : {reply.stop_reason}")
print(f"tokens      : {reply.usage.input_tokens} in / {reply.usage.output_tokens} out")

I'm sorry to hear about that! Here's what you can do:

First, check if the hospital was in our network (Article 2). We have 4,200 network hospitals across Tamil Nadu offering cashless treatment.

If your cashless request was declined, you'll need to pay the bill yourself and file for reimbursement within 30 days (Article 1). Please submit:
- Discharge summary
- Final bill
- Payment receipt

Contact our team with these documents, and we'll process your claim promptly. Is there anything else I can help clarify?

------------------------------------------------------------
stop_reason : end_turn
tokens      : 215 in / 128 out


**Expected result:** a short, warm reply that tells the customer to pay and file for reimbursement within 30 days with the three named documents, citing Article 1.

**Read the token count.** That is your bill. We sent Claude **3 articles**, not 4,000 — because the embedding model did the narrowing for free. That is the entire economic argument for this architecture, visible in one number.

---

### Part C — the negation trap, and Claude fixing it

**Objective:** demonstrate the Section 4 warning with real numbers, then fix it.

Cosine similarity measures *topic*, not *agreement*. Watch it rank a document that says the exact opposite of what the customer needs.

In [14]:
question = "Is dental treatment covered by my policy?"

candidates = [
    "Dental treatment is covered under this policy up to Rs 25,000 per year.",
    "Dental treatment is excluded from this policy and is not covered.",
]

q_vec = embedder.encode(question)
for doc in candidates:
    print(f"  cosine {float(embedder.encode(doc) @ q_vec):.3f}  |  {doc}")

print("\nBoth score high, and they say OPPOSITE things.")
print("Cosine similarity cannot tell 'covered' from 'not covered' -")
print("they are the same TOPIC. This is a real production failure mode.")

  cosine 0.701  |  Dental treatment is covered under this policy up to Rs 25,000 per year.
  cosine 0.774  |  Dental treatment is excluded from this policy and is not covered.

Both score high, and they say OPPOSITE things.
Cosine similarity cannot tell 'covered' from 'not covered' -
they are the same TOPIC. This is a real production failure mode.


In [15]:
# Now hand the same two documents to Claude and ask which one actually applies.
docs_text = "\n".join(f"{i+1}. {d}" for i, d in enumerate(candidates))

check = client.messages.create(
    model=MODEL,
    max_tokens=200,
    system="You check whether retrieved documents actually answer the question. Be brief and decisive.",
    messages=[{
        "role": "user",
        "content": (
            f"QUESTION: {question}\n\nRETRIEVED DOCUMENTS:\n{docs_text}\n\n"
            "These two documents contradict each other. Say which one answers the "
            "question and flag the contradiction in one sentence."
        )
    }],
)

print(check.content[0].text)

**Document 1 answers the question** – it provides specific coverage details (Rs 25,000/year). **Flag: These documents directly contradict each other; Document 2 states dental treatment is excluded while Document 1 states it's covered up to Rs 25,000 annually.**


**Expected result:** Claude notices the contradiction immediately and flags it. The embedding model could not.

### 🎯 The lesson of this lab, in one line

> **Embeddings are a cheap filter that gets you close. Claude is expensive judgment that gets you correct. Production systems use embeddings to narrow thousands of documents to five, and Claude to turn those five into a trustworthy answer.**

### 🧪 Try this

1. Change `customer_message` to something the knowledge base genuinely does not cover — "can I insure my car with you?" — and check that Claude says it does not know instead of inventing an answer. **This is the single most important behaviour to verify in any RAG system.**
2. Set `max_tokens=50` and watch `stop_reason` change from `end_turn` to `max_tokens`. Your code must handle that.
3. Retrieve `top_1` instead of `top_3` and see whether the answer gets worse. This is how you tune `k` — empirically, not by guessing.

---

# 🚀 Mini Project — the Kaveri Support Copilot

**Business case.** Kaveri Insurance's support desk handles 12,000 tickets a month. Priya's team estimates that **40%** are questions already answered in the help centre — customers simply cannot find the article, because they describe their problem in their own words. Every one of those becomes a human-handled ticket.

**Goal.** A search-and-answer assistant that finds the right article by *meaning* and drafts a cited reply.

**Architecture** (the diagram from the previous section):

```
  articles ─► chunk ─► MiniLM ─► ChromaDB
                                    │
  customer question ─► MiniLM ─► search top-5 ─► Claude Haiku ─► cited draft reply
```

### Build steps

| # | Step | Where |
|---|---|---|
| 1 | Write or gather 50 help-centre documents with metadata (category, region) | Day 10 |
| 2 | Chunk anything over ~200 words | Day 10 |
| 3 | Embed with MiniLM, store in ChromaDB with documents + metadata | Day 10 |
| 4 | Build keyword search too, so you can prove semantic search is better | Day 10 |
| 5 | Retrieve top-5 and pass to Claude Haiku with a strict "answer only from context" system prompt | Day 10 |
| 6 | Evaluate on 10 real customer phrasings | Day 10 |

### Definition of done

- [ ] 10 test queries written in **customer language**, not company language
- [ ] Semantic search beats keyword search on **recall@3** across those queries, with the numbers printed
- [ ] Claude's replies cite the article they used
- [ ] Claude says "I don't have that information" for an out-of-scope question — verified, not assumed
- [ ] You can explain every component to Priya, who is not technical

### Stretch

- Add a `category` metadata filter and show it improving precision
- Add hybrid search (keyword + semantic) and measure whether it helps
- Log every query and score to build a real evaluation set

**Tomorrow (Day 10) you build all of this.** Today's job was to make sure you understand *why* each piece is there.

---
---

# 📚 Revision Suite

---

## 1. Session Summary

Keyword search fails because it matches **letters**, not **meaning** — Priya's customer said "hospital bill was not paid" and the correct article said "cashless request declined", with zero words in common. The fix is to convert every piece of text into an **embedding**: a fixed-length list of numbers that acts as a **coordinate for meaning**, so that text meaning similar things lands in similar places. We compare those coordinates with **cosine similarity**, which measures the *angle* between two vectors and ignores their length — because a longer document is not a more meaningful one. Two models do almost all of this work in practice: **MiniLM** (384 dimensions, fast) and **MPNet** (768 dimensions, better), and both **silently truncate** long input, which is why chunking is mandatory rather than optional. A **vector database** then stores those vectors alongside the documents and metadata, and uses **approximate search** (IVF, HNSW) to skip most of the corpus — trading a measurable amount of recall for a large amount of speed. Finally, because cosine similarity measures topic and not agreement, the last step belongs to **Claude**: retrieve cheaply with embeddings, reason expensively and accurately with the LLM.

---

## 2. What You Learned Today

You can now:

- [ ] Explain an embedding as a **coordinate for meaning** using the latitude/longitude analogy
- [ ] Explain why keyword search returns nothing for "my hospital bill was not paid"
- [ ] Describe the four steps: tokenise → encode → pool → normalise
- [ ] State why input length does not change output vector size
- [ ] Compute cosine similarity by hand from the dot product and magnitudes
- [ ] Explain why cosine beats Euclidean distance for text
- [ ] Prove that on unit vectors, L2 and cosine rank identically
- [ ] Explain why "0.8 is a good score" is not a real rule
- [ ] Explain why high similarity does **not** mean agreement (the negation trap)
- [ ] Choose between MiniLM and MPNet with a numeric justification
- [ ] Explain silent truncation and why chunking is mandatory
- [ ] Explain what a vector database adds over a NumPy array — all five things
- [ ] Explain ANN, recall@k, and the speed/accuracy tradeoff
- [ ] Explain why FAISS is a library and Chroma is a database
- [ ] Draw the offline/online architecture from memory
- [ ] Write a Claude Haiku call that answers only from retrieved context

---

## 3. AI Architect Cheat Sheet

### The definitions

| Term | One-line definition |
|---|---|
| **Embedding** | A fixed-length list of numbers that is a coordinate for the meaning of a text |
| **Dimension** | How many numbers in that list (384 for MiniLM, 768 for MPNet) |
| **Cosine similarity** | The angle between two vectors, from -1 to 1; ignores length |
| **Dot product** | Multiply position by position and sum; equals cosine when vectors are unit length |
| **Chunking** | Splitting long documents so nothing gets silently truncated |
| **ANN** | Approximate Nearest Neighbour — skip most vectors, accept some misses |
| **recall@k** | Of the k truly-nearest documents, how many the index actually returned |
| **HNSW / IVF** | The two ANN index families: graph-walking / cluster-skipping |
| **Vector database** | A database whose WHERE clause is "nearest to this meaning" |
| **RAG** | Retrieve documents with embeddings, then have an LLM answer from them |

### The numbers worth memorising

| Fact | Value |
|---|---|
| MiniLM dimensions / params / truncation limit | 384 / 22.7M / 256 word pieces |
| MPNet dimensions / params / truncation limit | 768 / ~110M / 384 word pieces |
| Memory for 1M vectors at 384-d, float32 | ~1.5 GB |
| Memory for 1M vectors at 768-d, float32 | ~3 GB |
| Brute force is fine up to about | 100,000 documents |
| Typical cosine for *unrelated* real sentences | 0.0 – 0.3 (not -1) |
| FAISS released | 22 February 2017, Meta AI Research |
| ChromaDB open-sourced | 14 February 2023 |
| Claude Haiku model ID | `claude-haiku-4-5-20251001` |

### Decision table

| Situation | Choose |
|---|---|
| < 100k docs, prototype | NumPy array, or ChromaDB for the convenience |
| Need metadata filters and persistence, no server | **ChromaDB** |
| Already run Postgres | **pgvector** |
| Millions of vectors, need max control | **FAISS** (plus your own storage) |
| Millions of vectors, need a service | Qdrant / Weaviate / Milvus / Pinecone |
| Speed matters most | **MiniLM** (384-d) |
| Quality matters most | **MPNet** (768-d), or a hosted model like Voyage |
| Query is an ID, code or exact name | **Keyword search**, or hybrid |
| Answer must respect negation | Add a **reranker** or let **Claude** decide |

### Claude quick reference

```python
import anthropic
client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5-20251001"

reply = client.messages.create(
    model=MODEL,
    max_tokens=400,
    system="Answer ONLY from the provided context. If it is not there, say so.",
    messages=[{"role": "user", "content": f"CONTEXT:\n{context}\n\nQUESTION:\n{q}"}],
)

reply.content[0].text     # the answer
reply.stop_reason         # "end_turn" = finished, "max_tokens" = cut off
reply.usage.input_tokens  # your bill
```

---

## 4. Five-Minute Revision Guide

Read this before an interview. Nothing else.

1. **Keyword search matches letters. Semantic search matches meaning.** "Hospital bill not paid" vs "cashless request declined": zero shared words, identical meaning.
2. **An embedding is a coordinate for meaning.** Like latitude/longitude, but for sentences. Similar meaning → similar coordinates.
3. **Text in, fixed-size vector out.** Tokenise → Transformer encodes in context → mean-pool into one vector → normalise to length 1. A 3-word query and a 900-word document both give 384 numbers.
4. **Individual dimensions mean nothing.** Meaning lives across all of them together. Never say "dimension 7 is sentiment".
5. **Cosine similarity = angle between vectors.** Ignores length, because long is not the same as meaningful. Formula: dot product divided by the two magnitudes.
6. **On unit vectors, dot product = cosine, and L2 ranks identically.** `‖A−B‖² = 2 − 2cos`.
7. **In practice scores run 0 to 1, not -1 to 1.** Unrelated text lands around 0.0–0.3. Thresholds must be calibrated per model, per domain.
8. **High similarity ≠ agreement.** "Covered" and "not covered" score high together. Fix with a reranker or Claude.
9. **Models silently truncate.** MiniLM at 256 word pieces, MPNet at 384. No error. Chunk your documents.
10. **One embedding model for the whole index, forever.** Changing it means a full re-index.
11. **A vector database adds five things:** approximate search, persistence, documents+metadata, filters, and safe incremental updates. Only the first is about speed.
12. **ANN trades recall for speed — always quote both.** FAISS `nprobe`, HNSW `ef_search`.
13. **FAISS is a library, Chroma is a database.** FAISS returns row numbers; Chroma returns your documents.
14. **Under 100k documents you do not need ANN.** Knowing this is a seniority signal.
15. **The architecture is: retrieve cheap, reason expensive.** Embeddings narrow a million to five; Claude Haiku turns five into a cited answer.

---

## 5. Interview Preparation Notes

**Q1. What is an embedding?**
A fixed-length list of numbers representing the meaning of a text. Like latitude and longitude for a city, but for meaning: texts that mean similar things get similar coordinates, so similarity becomes arithmetic.

**Q2. Why does semantic search beat keyword search?**
Keyword search compares strings; it cannot connect "hospital bill not paid" to "cashless request declined" because they share no words. Embeddings compare meaning, so those two land next to each other.

**Q3. What is cosine similarity and why not Euclidean distance?**
Cosine measures the angle between two vectors and ignores their length. For text that is what we want: a longer document is not a more meaningful one. Note that if vectors are normalised — which MiniLM's and MPNet's are — Euclidean and cosine give the identical ranking, since ‖A−B‖² = 2 − 2cos.

**Q4. Your search returns a document stating the opposite of the answer. Why?**
Cosine similarity captures topic, not stance. "Covered" and "not covered" are the same topic. Fix with a cross-encoder reranker on the top 20, metadata filters, or by letting Claude judge which retrieved document actually answers the question.

**Q5. MiniLM or MPNet?**
Constraints first. MiniLM: 384-d, 22.7M parameters, fast, ideal for large corpora on CPU. MPNet: 768-d, ~110M parameters, better quality, twice the memory. I would benchmark both on 30–50 real queries from our own domain and pick on measured recall, not on a public leaderboard.

**Q6. What is chunking and why is it mandatory?**
Embedding models truncate silently — MiniLM at 256 word pieces, MPNet at 384. A long document embedded whole is represented by its opening paragraph only, and nothing errors. Chunking to roughly 200 words with a small overlap prevents this.

**Q7. What does a vector database give me over a NumPy array?**
Approximate search at scale, persistence, storage of documents and metadata alongside vectors, metadata filtering, and safe incremental updates. Below about 100,000 documents, a NumPy array is genuinely a reasonable production choice.

**Q8. Explain ANN and its cost.**
Approximate Nearest Neighbour skips most of the corpus using clustering (IVF) or a navigable graph (HNSW). The cost is recall — it can miss true nearest neighbours. You measure recall@k and tune `nprobe` or `ef_search` until you hit your target. Quoting latency without recall is meaningless.

**Q9. FAISS or ChromaDB?**
Different categories. FAISS is a library: an in-memory index returning row numbers, no metadata, no persistence unless you save the file. Chroma is an embedded database: documents, metadata, filters and persistence included. For a RAG prototype, Chroma. For maximum control at scale with your own storage layer, FAISS.

**Q10 (architecture). Design semantic search for 50 million documents at 500 QPS.**
Chunk to ~200 words, embed with MiniLM offline in a batch pipeline. 50M × 384 × 4 bytes ≈ 75 GB, so shard across nodes or apply product quantisation. HNSW for stable low latency at high QPS. Only the query is embedded online. Set a recall@10 target first — say 95% — then tune the index to meet it and measure p99 latency against it.

**Q11 (architecture). What breaks when you change the embedding model?**
The entire index. Vectors from different models are not comparable and often not even the same size. It requires a full re-index, so treat model choice as a long-lived architectural commitment and plan a blue/green index switch.

**Q12 (FDE). The client says "just use ChatGPT/Claude to search our documents."**
It does not scale or fit. A million documents will not fit in any context window, and paying an LLM to read them on every query is orders of magnitude more expensive than an embedding lookup. The standard pattern is retrieve-then-reason: embeddings narrow a million to five for fractions of a cent, then Claude reasons over those five. I would demo both side by side with their real documents and their real cost numbers.

---

## 6. Assignment

**Beginner.** Write 8 sentences: 4 about insurance claims and 4 about food. Embed them all, print the full 8×8 similarity matrix, and confirm the two topics form two blocks of high scores. Write one paragraph explaining what you see.

**Intermediate.** Take 5 real questions in *your own* words about any domain you know, and 15 documents written in *formal* language. Run keyword search and semantic search on both. Build a table of which method found the right document. Report recall@3 for each.

**Advanced.** Measure the truncation limit yourself. Embed the same key sentence buried at position 50, 200, 400 and 800 words into a filler document, and plot similarity to a query about that sentence. Find where the score collapses for MiniLM and for MPNet. Compare against the documented 256 and 384 word-piece limits.

**Project.** Build the Kaveri Support Copilot end-to-end using Day 10's notebook. Deliver: the ChromaDB collection, a keyword baseline, a 10-query evaluation table with recall@3 for both methods, and Claude-generated cited replies. Include one out-of-scope query proving Claude declines to answer.

---

## 7. Assessment

### Part A — Multiple choice (10)

1. An embedding is best described as:
   (a) a compressed copy of the text  (b) a coordinate for meaning  (c) a keyword list  (d) an encrypted string

2. `all-MiniLM-L6-v2` produces how many numbers per text?
   (a) 128  (b) 384  (c) 768  (d) depends on text length

3. Cosine similarity ignores:
   (a) direction  (b) vector length  (c) dimensions  (d) word order

4. A 3-word query and a 900-word document embedded with MPNet give vectors of size:
   (a) 3 and 900  (b) 768 and 768  (c) 384 and 768  (d) unpredictable

5. Two unrelated English sentences embedded with MiniLM will typically score around:
   (a) -1.0  (b) -0.5  (c) 0.0 to 0.3  (d) 0.9

6. "Dental is covered" vs "Dental is not covered" will have cosine similarity that is:
   (a) near -1  (b) near 0  (c) high  (d) exactly 0

7. FAISS is:
   (a) a managed cloud database  (b) a similarity-search library  (c) an embedding model  (d) a Postgres extension

8. What does `nprobe` control in FAISS IVF?
   (a) vector dimensions  (b) how many clusters are searched  (c) how many results are returned  (d) the number of threads

9. You feed a 3,000-word document to MiniLM without chunking. What happens?
   (a) an error is raised  (b) it is summarised  (c) it is silently truncated at ~256 word pieces  (d) the vector gets longer

10. You switch the embedding model in production. What must you do?
   (a) nothing  (b) re-embed queries only  (c) re-index the entire corpus  (d) increase `n_results`

### Part B — Short answer (5)

11. Explain cosine similarity to someone who has never seen a vector, in three sentences.
12. Why does search quality depend more on the embedding model than on the vector database?
13. Give a concrete query where keyword search beats semantic search, and explain why.
14. What is recall@k, and why is a latency number meaningless without it?
15. Why does the architecture put embeddings *before* Claude instead of after?

### Part C — Scenario (3)

16. Your semantic search over 200,000 policy documents returns plausible-looking but wrong articles for about 30% of queries. The embedding model is MiniLM, documents average 4,000 words and were embedded whole. Diagnose the most likely cause and give your fix order.

17. A bank wants semantic search over 8 million transactions, and queries include things like "payment to Reliance in March" and also exact reference numbers like "TXN-88-2291". Design the retrieval layer and justify it.

18. Your ANN index halved p99 latency, but the support team reports search "got worse" the same week. What single metric do you check first, what do you expect to find, and what do you change?

---

## 8. Answer Key

**Part A:** 1-(b) · 2-(b) · 3-(b) · 4-(b) · 5-(c) · 6-(c) · 7-(b) · 8-(b) · 9-(c) · 10-(c)

**Part B**

11. Every sentence becomes an arrow pointing somewhere in space. Cosine similarity measures the angle between two arrows: 1 means they point the same way, 0 means they are unrelated. It deliberately ignores how long the arrows are, because a longer document is not a more meaningful one.

12. The embedding model decides *where* each document lands in meaning space — that is the actual quality of the search. The database only decides how fast you can find nearby points and how well you can operate the system. Swapping databases does not move any document closer to the right query.

13. Any exact identifier: "policy KVI-2024-88123", "error code 0x8007", "Ramesh Kumar". Keyword search matches the string exactly. Semantic search will happily return *other* policy numbers because they look and mean roughly the same thing — which is exactly wrong. Hence hybrid search in production.

14. recall@k is the fraction of the truly-nearest k documents that the index actually returned. Approximate indexes buy speed by skipping vectors, so latency and recall move in opposite directions — a claim of "2 ms search" is meaningless if it comes with 55% recall, because that is a broken search engine that happens to be fast.

15. Cost and capacity. Embeddings narrow a million documents to five for a fraction of a cent and a few milliseconds. Claude cannot fit a million documents in its context window, and reading them all on every query would cost orders of magnitude more. Cheap narrowing first, expensive intelligence last.

**Part C**

16. **Most likely cause: silent truncation.** 4,000-word documents embedded whole with MiniLM are represented by their first ~200 words only, so every vector encodes boilerplate headers rather than content — which is exactly what produces plausible-looking but wrong results. **Fix order:** (1) chunk to ~200 words with ~20-word overlap and re-index — this alone usually fixes most of it; (2) build a 30-query evaluation set so improvement is measured, not felt; (3) add a cross-encoder reranker on the top 20; (4) only then consider moving to MPNet or a hosted model. Note the order: fix the data pipeline before upgrading the model.

17. **Hybrid retrieval.** Reference numbers must go through an exact/keyword path — semantic search on "TXN-88-2291" will return other transaction IDs, which is a correctness failure, not a ranking one. Natural-language queries go through the embedding path. Route by pattern (a regex for ID-shaped queries) or run both and blend scores with reciprocal rank fusion. Merchant names and dates should be **metadata filters**, not semantic content, so "in March" becomes a structured constraint. At 8 million vectors, HNSW in a server-based store (Qdrant/Milvus) or pgvector if the bank already runs Postgres — and Postgres is likely the right answer, because the transaction data already lives there and data residency and audit are non-negotiable in banking.

18. **Check recall@k first** — specifically recall@10 of the ANN index against an exact brute-force baseline on a sample of real queries. You expect to find it has dropped well below your target, because the index is now skipping clusters that held correct results. **Change:** raise `nprobe` (IVF) or `ef_search` (HNSW) until recall is back at target, then re-measure p99. If you cannot get both, you need more shards or a better index, not a different threshold. The underlying lesson: latency was tuned without a recall guardrail, so add recall to the deployment checks permanently.

---

## 9. Sources (every fact in this notebook is checkable)

- **MiniLM model card** (384 dims, 22.7M params, 256 word-piece truncation, 1,170,060,424 training pairs, TPU v3-8 sprint) — [huggingface.co/sentence-transformers/all-MiniLM-L6-v2](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)
- **MPNet model card** (768 dims, 1B pairs, 384 word-piece truncation, base `microsoft/mpnet-base`) — [huggingface.co/sentence-transformers/all-mpnet-base-v2](https://huggingface.co/sentence-transformers/all-mpnet-base-v2)
- **Anthropic on embeddings** (Anthropic does not offer an embedding model; recommends Voyage AI; Voyage vectors normalised so dot product = cosine) — [platform.claude.com/docs/en/build-with-claude/embeddings](https://platform.claude.com/docs/en/build-with-claude/embeddings)
- **FAISS announcement** (Meta AI Research, 2017; ~8.5× faster than prior state of the art; first billion-vector k-NN graph) — [engineering.fb.com/2017/03/29/data-infrastructure/faiss-a-library-for-efficient-similarity-search](https://engineering.fb.com/2017/03/29/data-infrastructure/faiss-a-library-for-efficient-similarity-search/)
- **FAISS repository and index documentation** — [github.com/facebookresearch/faiss](https://github.com/facebookresearch/faiss)
- **ChromaDB documentation** — [docs.trychroma.com](https://docs.trychroma.com/)
- **Sentence-Transformers pretrained models** — [sbert.net](https://www.sbert.net/docs/sentence_transformer/pretrained_models.html)

---

### ▶️ Next: **Day 10 — Build a Semantic Search Engine over 50 Documents with ChromaDB**

Tomorrow you build everything above, compare keyword against semantic search with real numbers, and ship the Kaveri Support Copilot.